<a href="https://colab.research.google.com/github/SiriChandanaMannem/Brain-Tumor-Detection-/blob/main/brain_tumor_detection_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

!pip install gradio tensorflow


In [5]:

from google.colab import files

uploaded = files.upload()


Saving Y1.jpg to Y1 (1).jpg


In [ ]:

from IPython.display import Image, display
import os

for file in ["Y1.jpg", "Y1 (1).jpg"]:
    if os.path.exists(file):
        print("Displaying:", file)
        display(Image(filename=file))


In [8]:

import gradio as gr
import tensorflow as tf
import numpy as np

# Load your trained CNN model
MODEL_PATH = "brain_tumor_cnn.keras"

try:
    model = tf.keras.models.load_model(MODEL_PATH)
except Exception:
    model = None

# Demo credentials — change these before deployment
USERNAME = "admin"
PASSWORD = "python123"


def login(username, password):
    if username == USERNAME and password == PASSWORD:
        return (
            gr.update(visible=False),
            gr.update(visible=True),
            "Login successful!"
        )

    return (
        gr.update(visible=True),
        gr.update(visible=False),
        "Invalid username or password."
    )


def predict(image):
    if image is None:
        return "Please upload an MRI image."

    if model is None:
        return (
            "Model not loaded. Add brain_tumor_cnn.keras "
            "to the project folder and restart the app."
        )

    image = tf.image.resize(image, (180, 180))
    image = tf.expand_dims(image, axis=0)

    probability = float(model.predict(image, verbose=0)[0][0])

    if probability >= 0.5:
        return f"Model prediction: YES (tumor class). Score: {probability:.2%}"

    return f"Model prediction: NO (no-tumor class). Score: {1 - probability:.2%}"


with gr.Blocks(title="Brain Tumor Detection") as app:
    gr.Markdown("# Brain Tumor Detection from MRI Images")

    with gr.Group(visible=True) as login_page:
        gr.Markdown("## Login")
        username = gr.Textbox(label="Username")
        password = gr.Textbox(label="Password", type="password")
        login_button = gr.Button("Login")
        login_message = gr.Textbox(label="Status", interactive=False)

    with gr.Group(visible=False) as prediction_page:
        gr.Markdown("## Upload MRI Image")
        mri_image = gr.Image(type="numpy", label="MRI Image")
        predict_button = gr.Button("Predict")
        result = gr.Textbox(label="Prediction Result")
        logout_button = gr.Button("Logout")

    login_button.click(
        login,
        inputs=[username, password],
        outputs=[login_page, prediction_page, login_message]
    )

    predict_button.click(
        predict,
        inputs=mri_image,
        outputs=result
    )

    logout_button.click(
        lambda: (
            gr.update(visible=True),
            gr.update(visible=False),
            "",
            None,
            ""
        ),
        outputs=[
            login_page,
            prediction_page,
            username,
            mri_image,
            result
        ]
    )

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d0586231f56d953139.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:

import os

print("Files in Colab:", os.listdir("/content"))


Files in Colab: ['.config', 'Y1.jpg', '.gradio', 'Y1 (1).jpg', 'sample_data']


In [6]:

import os

print("Files in Colab:")
print(os.listdir("/content"))


Files in Colab:
['.config', 'Y1.jpg', 'Y1 (1).jpg', 'sample_data']
